In [ ]:
%pip install pandas numpy scikit-learn matplotlib
%pip install openpyxl

In [16]:
import json
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")  # put your three files in a "data" folder next to this script

# 1. Load files (default C engine; no rows are silently skipped)
environmental_df = pd.read_csv(DATA_DIR / "Environmental sensor.csv")
occupancy_df = pd.read_csv(DATA_DIR / "Occupancy sensor.csv")
sensor_locations_df = pd.read_excel(DATA_DIR / "Sensor ID and Locations.xlsx")

# Ensure the columns index and row index are flat and clean
environmental_df.columns = [str(col) for col in environmental_df.columns]
environmental_df.reset_index(drop=True, inplace=True)

# 2. Define the JSON parsing helper function
def parse_sensor_json(json_str):
    if pd.isna(json_str):
        return {}
    try:
        if isinstance(json_str, str):
            if json_str.startswith('""') or '""' in json_str:
                json_str = json_str.replace('""', '"')
            if json_str.startswith('"') and json_str.endswith('"'):
                json_str = json_str[1:-1]
            data = json.loads(json_str)
        else:
            data = json_str
            
        extracted = {}
        if isinstance(data, list):
            for item in data:
                var_info = item.get('variable', {})
                var_name = var_info.get('name')
                values_list = item.get('values', [])
                if values_list and isinstance(values_list, list):
                    metric_value = values_list[0].get('value')
                    if var_name is not None:
                        extracted[var_name] = metric_value
        return extracted
    except Exception:
        return {}

# 3. Parse and extract key-value metrics from environmental data
parsed_metrics = environmental_df['jsondata'].apply(parse_sensor_json)
metrics_df = pd.DataFrame(parsed_metrics.tolist())
metrics_df.reset_index(drop=True, inplace=True)

# Map standard headings to include units explicitly
unit_mapping = {
    'Battery': 'Battery (V)',
    'Carbon dioxide': 'Carbon dioxide (ppm)',
    'Humidity': 'Humidity (%)',
    'Formaldehyde': 'Formaldehyde (mg/m³)',
    'Pressure': 'Pressure (mb)',
    'Temperature': 'Temperature (°C)',
    'Light': 'Light (lx)',
    'Light Volatile Organic Compounds': 'Light Volatile Organic Compounds (ppb)',
    'Particulate matter 1': 'Particulate matter 1 (µg/m³)',
    'Particulate matter 4': 'Particulate matter 4 (µg/m³)',
    'Particulate matter 2.5': 'Particulate matter 2.5 (µg/m³)',
    'Particulate matter 10': 'Particulate matter 10 (µg/m³)'
}
metrics_df.rename(columns=unit_mapping, inplace=True)

# 4. Combine extracted metrics with the original DataFrame (dropping original raw JSON column)
environmental_expanded_df = pd.concat(
    [environmental_df.drop(columns=['jsondata']), metrics_df],
    axis=1
)

# 5. Display summaries and verify columns directly
print("--- Verified Column Names in Environmental Data ---")
for col in environmental_expanded_df.columns:
    print(f" - {col}")

print("\n--- Expanded Environmental Sensor Data with Units ---")
display(environmental_expanded_df.head())

print("\n--- Occupancy Sensor Data ---")
display(occupancy_df.head())

print("\n--- Sensor ID and Locations Data ---")
display(sensor_locations_df.head())

--- Verified Column Names in Environmental Data ---
 - id
 - sensorid
 - devicetype
 - status
 - processing_errors
 - createdate
 - processdate
 - Battery (V)
 - Carbon dioxide (ppm)
 - Humidity (%)
 - Formaldehyde (mg/m³)
 - Pressure (mb)
 - Temperature (°C)
 - Light Volatile Organic Compounds (ppb)
 - Particulate matter 1 (µg/m³)
 - Particulate matter 4 (µg/m³)
 - Particulate matter 2.5 (µg/m³)
 - Particulate matter 10 (µg/m³)

--- Expanded Environmental Sensor Data with Units ---


,id,sensorid,devicetype,status,processing_errors,createdate,processdate,Battery (V),Carbon dioxide (ppm),Humidity (%),Formaldehyde (mg/m³),Pressure (mb),Temperature (°C),Light Volatile Organic Compounds (ppb),Particulate matter 1 (µg/m³),Particulate matter 4 (µg/m³),Particulate matter 2.5 (µg/m³),Particulate matter 10 (µg/m³)
0,2,6012002000241,env,Processed,NaN,2024-02-29 03:11:58.210358+11,2024-02-29 03:11:58.210358+11,4.1875,603,66,6.0,1011,23.8,14.0,8.0,9.0,9.0,9.0
1,4,6012002000241,env,Processed,NaN,2024-02-29 17:09:08.833822+11,2024-02-29 17:09:08.833822+11,4.1875,603,66,6.0,1011,23.8,14.0,8.0,9.0,9.0,9.0
2,5,6012002000326,env,Processed,NaN,2024-02-29 17:09:12.444245+11,2024-02-29 17:09:12.444245+11,4.1875,525,68,5.0,1012,23.8,7.0,6.0,6.0,6.0,6.0
3,16,6012002000777,env,Processed,NaN,2024-02-29 17:10:24.443407+11,2024-02-29 17:10:24.443407+11,4.1562,425,51,1.0,1015,18.8,71.0,1.0,3.0,3.0,4.0
4,24,6012002000869,env,Processed,NaN,2024-02-29 17:11:11.348578+11,2024-02-29 17:11:11.348578+11,4.1250,465,56,17.0,1010,24.3,0.0,2.0,2.0,2.0,2.0



--- Occupancy Sensor Data ---


,id,deviceid,floorspaceid,occupancystatus,headcount,collecteddate,occupancystatuschangedate,previousoccupancystatus
0,136043,56923747-b390-4f2a-8930-808429132266,941f0352-bb69-47d0-a9c5-ac29eb4f7319,CurrentlyOccupied,1,2023-11-24 00:36:52+11,2023-11-24 00:36:52+11,NotOccupied
1,136048,56923747-b390-4f2a-8930-808429132266,941f0352-bb69-47d0-a9c5-ac29eb4f7319,RecentlyOccupied,0,2023-11-24 00:37:32+11,2023-11-24 00:37:32+11,CurrentlyOccupied
2,136050,56923747-b390-4f2a-8930-808429132266,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,CurrentlyOccupied,1,2023-11-24 00:37:32+11,2023-11-24 00:37:32+11,NotOccupied
3,136052,56923747-b390-4f2a-8930-808429132266,941f0352-bb69-47d0-a9c5-ac29eb4f7319,NotOccupied,0,2023-11-24 00:37:52+11,2023-11-24 00:37:52+11,RecentlyOccupied
4,136053,56923747-b390-4f2a-8930-808429132266,0f3f37f8-0335-41e0-ae87-a1d677dfc7d7,RecentlyOccupied,0,2023-11-24 00:37:52+11,2023-11-24 00:37:52+11,CurrentlyOccupied



--- Sensor ID and Locations Data ---


,Building,Floor/Room,Area,Physical Label,Sensor,ID
0,60,1-PS01,Corridor,ES1104,Env,6012003000486
1,69,1-PS02,Breakout,ES2113,Env,6012003000882
2,60,1.40,Board Room,ES1102,Env,6012003000462
3,60,1.41,Open Office,ES1110,Env,6012003000424
4,60,1.41A,Focus Room 1,ES1101,Env,6012003000448
